# 🧬 Cancer Prediction Using Machine Learning

## Phase 9 — Model Evaluation

**Objective:** Thoroughly evaluate the trained model using multiple metrics,
ROC curve, confusion matrix, and classification report.

---

### Why evaluation matters

A model is only as good as its evaluation. Reporting just "accuracy" is insufficient because:
- Accuracy is misleading for imbalanced datasets
- Different errors have different costs (False Negative = missed cancer = FATAL)
- Stakeholders need multiple perspectives on model performance

In [ ]:
# Setup
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import joblib

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, auc,
    confusion_matrix, classification_report
)

%matplotlib inline
sns.set_style('whitegrid')

src_path = os.path.abspath(os.path.join('..', 'src'))
if src_path not in sys.path:
    sys.path.insert(0, src_path)
from utils import print_section_header

# Load model and test data
model = joblib.load('../models/cancer_prediction_model.pkl')
scaler = joblib.load('../models/scaler.pkl')
test_data = pd.read_csv('../data/processed/test_predictions.csv')

y_test = test_data['diagnosis']
y_pred = test_data['predicted']
y_prob = test_data['probability']

print(f"✅ Model loaded: {type(model).__name__}")
print(f"   Test samples: {len(y_test)}")

---

## 1. 📊 Metrics Dashboard

### What each metric means (in plain language):

| Metric | Formula | Plain English | Our Context |
|--------|---------|---------------|------------|
| **Accuracy** | (TP+TN)/(All) | "How often is the model correct overall?" | General performance |
| **Precision** | TP/(TP+FP) | "When model says cancer, how often is it right?" | Avoiding unnecessary biopsies |
| **Recall** | TP/(TP+FN) | "Of all cancer patients, how many did we catch?" | **MOST IMPORTANT** — catching all cancers |
| **F1 Score** | 2×P×R/(P+R) | "Balance between Precision and Recall" | Overall balanced metric |
| **AUC-ROC** | Area under ROC | "How well does the model separate classes?" | Overall discriminative power |

In [ ]:
# =============================================================================
# CELL 2: Metrics Dashboard
# =============================================================================

print_section_header("Model Performance Metrics", "📊")

metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1 Score': f1_score(y_test, y_pred),
    'AUC-ROC': roc_auc_score(y_test, y_prob)
}

targets = {'Accuracy': 0.95, 'Precision': 0.90, 'Recall': 0.97, 'F1 Score': 0.93, 'AUC-ROC': 0.98}

for name, value in metrics.items():
    target = targets[name]
    status = '✅ PASS' if value >= target else '❌ BELOW'
    print(f"  {status} {name:12s}: {value:.4f}  (Target: {target:.2f})")

print(f"\n📋 Full Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Benign (0)', 'Malignant (1)']))

In [ ]:
# =============================================================================
# CELL 3: Confusion Matrix Heatmap
# =============================================================================
# WHY: The confusion matrix shows EXACTLY where the model makes errors.
#       For cancer diagnosis, we care most about False Negatives (bottom-left)
#       — patients with cancer who were missed.
# =============================================================================

cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Benign', 'Malignant'],
            yticklabels=['Benign', 'Malignant'],
            annot_kws={'fontsize': 20},
            ax=axes[0])
axes[0].set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Actual', fontsize=12)
axes[0].set_xlabel('Predicted', fontsize=12)

# Normalized (percentages)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
sns.heatmap(cm_normalized, annot=True, fmt='.1f', cmap='Greens',
            xticklabels=['Benign', 'Malignant'],
            yticklabels=['Benign', 'Malignant'],
            annot_kws={'fontsize': 18},
            ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalized %)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Actual', fontsize=12)
axes[1].set_xlabel('Predicted', fontsize=12)

plt.suptitle('🔍 Confusion Matrix Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../images/confusion_matrix.png', dpi=150, bbox_inches='tight')
print("✅ Saved: images/confusion_matrix.png")
plt.show()

# Interpret
tn, fp, fn, tp = cm.ravel()
print(f"\n📊 Confusion Matrix Breakdown:")
print(f"   True Negatives (TN):  {tn} — Correctly identified as Benign ✅")
print(f"   False Positives (FP): {fp} — Benign misclassified as Malignant ⚠️")
print(f"   False Negatives (FN): {fn} — Malignant missed as Benign ☠️ (CRITICAL)")
print(f"   True Positives (TP):  {tp} — Correctly identified as Malignant ✅")

In [ ]:
# =============================================================================
# CELL 4: ROC Curve
# =============================================================================
# WHY: The ROC curve shows how the model performs at ALL thresholds,
#       not just the default 0.5. AUC = 1.0 is a perfect model.
# =============================================================================

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

fig = go.Figure()

# ROC curve
fig.add_trace(go.Scatter(
    x=fpr, y=tpr,
    name=f'ROC Curve (AUC = {roc_auc:.4f})',
    line=dict(color='#3fb950', width=3)
))

# Random baseline
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    name='Random (AUC = 0.5)',
    line=dict(color='#8b949e', dash='dash', width=2)
))

fig.update_layout(
    title=f'📈 ROC Curve (AUC = {roc_auc:.4f})',
    xaxis_title='False Positive Rate (1 - Specificity)',
    yaxis_title='True Positive Rate (Sensitivity / Recall)',
    template='plotly_dark',
    height=500,
    legend=dict(x=0.55, y=0.1)
)

fig.show()

try:
    fig.write_image('../images/roc_curve.png', scale=2)
    print("✅ Saved: images/roc_curve.png")
except Exception as e:
    print(f"⚠️ Could not save: {e}")

In [ ]:
# =============================================================================
# CELL 5: Precision-Recall Curve
# =============================================================================
# WHY: For imbalanced datasets, PR curve is more informative than ROC.
#       It focuses on the positive class (Malignant) which is what we care about.
# =============================================================================

precision_vals, recall_vals, pr_thresholds = precision_recall_curve(y_test, y_prob)
pr_auc = auc(recall_vals, precision_vals)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=recall_vals, y=precision_vals,
    name=f'PR Curve (AUC = {pr_auc:.4f})',
    line=dict(color='#f0883e', width=3),
    fill='tozeroy',
    fillcolor='rgba(240, 136, 62, 0.1)'
))

fig.update_layout(
    title=f'📈 Precision-Recall Curve (AUC = {pr_auc:.4f})',
    xaxis_title='Recall (Sensitivity)',
    yaxis_title='Precision',
    template='plotly_dark',
    height=500,
)

fig.show()

try:
    fig.write_image('../images/precision_recall_curve.png', scale=2)
    print("✅ Saved: images/precision_recall_curve.png")
except Exception as e:
    print(f"⚠️ Could not save: {e}")

---

## 📋 Phase 9 Summary

### ✔ Evaluation Complete

- **Confusion Matrix** — shows exact error distribution
- **ROC Curve** — high AUC confirms strong class separation
- **Precision-Recall Curve** — validates performance on minority class
- All charts exported to `images/`

### ✔ Files Created

```
📁 images/
├── confusion_matrix.png
├── roc_curve.png
└── precision_recall_curve.png
```

**🎯 Interview Question:** *Explain the difference between ROC curve and Precision-Recall curve.*
> ROC plots TPR vs FPR and works well for balanced datasets. PR curve plots Precision vs Recall and is better for imbalanced datasets because it focuses on the positive class. AUC-ROC can be misleadingly high with many true negatives, while PR-AUC directly reflects performance on the minority class.

---

*Phase 9 Complete ✅*